# Cancer Type Classification using NLP
## TCGA Pathology Reports | 32 Cancer Types | 4 Models
### Random Forest · Linear SVM · Google BERT · PubMedBERT
#### 🔥 Optimized Version — 5 Epochs | Enhanced Settings

## Step 1 — Imports & Setup

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import os, re, random, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

import torch
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from datasets import Dataset

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

# Auto-detect best available device
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Torch version : {torch.__version__}')
print(f'Device        : {DEVICE}')
print('All imports successful!')


## Step 2 — Download Dataset
> Uses Python built-in `urllib`. Auto-skips if files already exist.

In [ ]:
if not os.path.exists('TCGA_Reports.csv'):
    print('Downloading TCGA_Reports.csv.zip ...')
    urllib.request.urlretrieve(
        'https://github.com/tatonetti-lab/tcga-path-reports/raw/main/TCGA_Reports.csv.zip',
        'TCGA_Reports.csv.zip'
    )
    with zipfile.ZipFile('TCGA_Reports.csv.zip', 'r') as z:
        z.extractall('.')
    print('Done — extracted TCGA_Reports.csv')
else:
    print('TCGA_Reports.csv already exists, skipping.')


In [ ]:
if not os.path.exists('tcga_patient_to_cancer_type.csv'):
    print('Downloading cancer type labels ...')
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/tatonetti-lab/tcga-path-reports'
        '/main/data/tcga_metadata/tcga_patient_to_cancer_type.csv',
        'tcga_patient_to_cancer_type.csv'
    )
    print('Done.')
else:
    print('tcga_patient_to_cancer_type.csv already exists, skipping.')


## Step 3 — Load & Merge Data

In [ ]:
reports      = pd.read_csv('TCGA_Reports.csv')
reports['patient_id'] = reports['patient_filename'].str[:12]
cancer_types = pd.read_csv('tcga_patient_to_cancer_type.csv')

data = pd.merge(reports, cancer_types, on='patient_id')
data = data[['text', 'cancer_type']].copy().reset_index(drop=True)

print(f'Total samples : {len(data)}')
print(f'Cancer types  : {data["cancer_type"].nunique()}')
data.head()


## Step 4 — Exploratory Data Analysis (EDA)

In [ ]:
counts = data['cancer_type'].value_counts()
plt.figure(figsize=(15, 5))
sns.barplot(x=counts.index, y=counts.values,
            hue=counts.index, palette='viridis', legend=False)
plt.title('Reports per Cancer Type', fontsize=14)
plt.xlabel('Cancer Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print(f'Top 5:\n{counts.head()}')
print(f'\nBottom 5:\n{counts.tail()}')


## Step 5 — Text Cleaning

In [ ]:
def clean_text(text):
    text  = text.lower()
    text  = re.sub(r'[^a-zA-Z ]', ' ', text)
    words = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return ' '.join(words)

data['clean_text'] = data['text'].apply(clean_text)
data = data[data['clean_text'].str.strip() != ''].reset_index(drop=True)
print(f'Samples after cleaning: {len(data)}')
data[['cancer_type', 'clean_text']].head(3)


## Step 6 — Label Encoding

In [ ]:
encoder    = LabelEncoder()
data['label'] = encoder.fit_transform(data['cancer_type'])
num_labels = len(encoder.classes_)
print(f'Cancer types (num_labels): {num_labels}')
print(f'Classes: {list(encoder.classes_)}')


## Step 7 — Train / Test Split (Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data['clean_text'], data['cancer_type'],
    test_size=0.2, random_state=42,
    stratify=data['cancer_type']   # ensures all 32 types in both sets
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')


## Step 8 — TF-IDF Vectorization
> 🔥 **Upgraded:** `max_features` increased from 10,000 → 15,000 for richer vocabulary coverage.

In [ ]:
# Upgraded: 15,000 features (was 10,000) — captures more rare medical terms
vectorizer  = TfidfVectorizer(
    max_features=15000,     # increased from 10000 for better vocab coverage
    ngram_range=(1, 2),     # unigrams + bigrams
    sublinear_tf=True,      # log-scale term frequency
    min_df=2,               # ignore terms appearing in fewer than 2 docs
    max_df=0.95             # ignore terms appearing in more than 95% of docs
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)
print(f'Train matrix: {X_train_vec.shape}')
print(f'Test  matrix: {X_test_vec.shape}')


## Step 9 — Train Random Forest
> 🔥 **Upgraded:** Trees increased from 200 → 300, added `min_samples_leaf=2` to reduce overfitting.

In [ ]:
print('Training Random Forest ...')
# Upgraded: 300 trees (was 200), min_samples_leaf prevents overfitting
rf_model = RandomForestClassifier(
    n_estimators=300,       # increased from 200
    min_samples_leaf=2,     # each leaf needs at least 2 samples — reduces overfitting
    max_features='sqrt',    # use sqrt(features) per split — standard best practice
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train_vec, y_train)
rf_pred  = rf_model.predict(X_test_vec)
rf_acc   = accuracy_score(y_test, rf_pred)
rf_f1    = f1_score(y_test, rf_pred, average='weighted', zero_division=0)
print(f'Random Forest — Accuracy: {rf_acc:.4f} ({rf_acc*100:.2f}%) | F1: {rf_f1:.4f}')


## Step 10 — Train Linear SVM
> 🔥 **Upgraded:** `C` tuned from 1.0 → 0.5 for better generalization on 32-class problem.

In [ ]:
print('Training Linear SVM ...')
# Upgraded: C=0.5 (was 1.0) — slightly lower C = better generalization
# max_iter increased to 5000 to ensure full convergence
svm_model = LinearSVC(
    C=0.5,              # tuned from 1.0 — reduces margin errors on hard cases
    max_iter=5000,      # increased from 2000 — ensures convergence
    random_state=42
)
svm_model.fit(X_train_vec, y_train)
svm_pred  = svm_model.predict(X_test_vec)
svm_acc   = accuracy_score(y_test, svm_pred)
svm_f1    = f1_score(y_test, svm_pred, average='weighted', zero_division=0)
print(f'Linear SVM — Accuracy: {svm_acc:.4f} ({svm_acc*100:.2f}%) | F1: {svm_f1:.4f}')


## Step 11 — Classical Model Evaluation

In [ ]:
for name, acc, f1, pred in [
    ('RANDOM FOREST', rf_acc,  rf_f1,  rf_pred),
    ('LINEAR SVM',    svm_acc, svm_f1, svm_pred)
]:
    print('=' * 65)
    print(f'{name} — FULL EVALUATION')
    print('=' * 65)
    print(f'Accuracy : {acc:.4f} ({acc*100:.2f}%)')
    print(f'F1 Score : {f1:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, pred, zero_division=0))


## Step 12 — Confusion Matrices (Classical Models)

In [ ]:
label_names = list(encoder.classes_)
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, preds, title, cmap in zip(
    axes,
    [rf_pred, svm_pred],
    ['Random Forest', 'Linear SVM'],
    ['Blues', 'Greens']
):
    cm = confusion_matrix(y_test, preds, labels=label_names)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=label_names, yticklabels=label_names,
                ax=ax, linewidths=0.3, annot_kws={'size': 7})
    ax.set_title(f'{title} — Confusion Matrix', fontsize=13)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)
plt.tight_layout()
plt.show()


## Step 13 — Prepare Data for Transformer Models

In [ ]:
bert_data = data[['clean_text','label']].copy()
bert_data = bert_data.rename(columns={'clean_text':'text'})
bert_data['label'] = bert_data['label'].astype(int)

bert_train_df, bert_test_df = train_test_split(
    bert_data, test_size=0.2, random_state=42,
    stratify=bert_data['label']
)
bert_train_df = bert_train_df.reset_index(drop=True)
bert_test_df  = bert_test_df.reset_index(drop=True)
print(f'BERT Train: {len(bert_train_df)} | Test: {len(bert_test_df)} | Labels: {num_labels}')


## Step 14 — compute_metrics Function

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted', zero_division=0)
    }
print('compute_metrics ready.')


## Step 15 — Training Arguments
> 🔥 **Key upgrades:**
> - `num_train_epochs` increased **3 → 5** for higher accuracy
> - `learning_rate` reduced **2e-5 → 1e-5** so model learns more carefully at more epochs
> - `warmup_ratio` increased **0.1 → 0.15** for smoother warm-up over 5 epochs
> - `weight_decay` increased **0.01 → 0.02** to prevent overfitting at more epochs
> - `early_stopping_patience` increased **2 → 3** to give model enough time to improve

In [ ]:
def make_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,

        # ── Core training ──────────────────────────────────
        learning_rate=1e-5,             # reduced from 2e-5 — more careful learning over 5 epochs
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,             # INCREASED from 3 — sweet spot for this dataset size

        # ── Evaluation & saving ────────────────────────────
        eval_strategy='epoch',          # evaluate after every epoch
        save_strategy='epoch',          # save checkpoint after every epoch
        load_best_model_at_end=True,    # auto-keep the best checkpoint
        metric_for_best_model='accuracy',
        greater_is_better=True,

        # ── Regularization ─────────────────────────────────
        warmup_ratio=0.15,              # increased from 0.1 — smoother warm-up over 5 epochs
        weight_decay=0.02,              # increased from 0.01 — stronger regularization

        # ── Logging ────────────────────────────────────────
        logging_steps=50,
        fp16=False,                     # MPS does not support fp16
        report_to='none',               # disable wandb
    )

print('Training args ready.')
print('Epochs      : 5  (was 3)')
print('LR          : 1e-5  (was 2e-5)')
print('Warmup      : 15%  (was 10%)')
print('Weight decay: 0.02  (was 0.01)')


## Step 16 — Google BERT: Tokenize

In [ ]:
print('Loading BERT tokenizer ...')
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_bert(batch):
    return bert_tokenizer(
        batch['text'], truncation=True,
        padding='max_length', max_length=256
    )

bert_train_ds = Dataset.from_pandas(bert_train_df)
bert_test_ds  = Dataset.from_pandas(bert_test_df)
bert_train_ds = bert_train_ds.map(tokenize_bert, batched=True)
bert_test_ds  = bert_test_ds.map(tokenize_bert,  batched=True)
bert_train_ds = bert_train_ds.rename_column('label', 'labels')
bert_test_ds  = bert_test_ds.rename_column('label',  'labels')
cols = ['input_ids', 'attention_mask', 'token_type_ids', 'labels']
bert_train_ds.set_format(type='torch', columns=cols)
bert_test_ds.set_format( type='torch', columns=cols)
print(f'BERT tokenization done. Train: {len(bert_train_ds)} | Test: {len(bert_test_ds)}')


## Step 17 — Train Google BERT (5 Epochs)
> ⏱️ ~35–60 min on GPU &nbsp;|&nbsp; ~3–5 hrs on Apple Silicon &nbsp;|&nbsp; ~10+ hrs on CPU

> 💡 **Recommended:** Run on Google Colab with free T4 GPU for best speed.

In [ ]:
print('Loading Google BERT model ...')
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=num_labels
)

bert_trainer = Trainer(
    model=bert_model,
    args=make_training_args('./bert_results'),
    train_dataset=bert_train_ds,
    eval_dataset=bert_test_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # increased from 2
)

print('Training Google BERT — 5 epochs ...')
bert_trainer.train()
print('Google BERT training complete!')


## Step 18 — Evaluate Google BERT

In [ ]:
bert_results = bert_trainer.evaluate()
bert_acc  = bert_results.get('eval_accuracy', 0)
bert_f1   = bert_results.get('eval_f1', 0)
bert_loss = bert_results.get('eval_loss', 0)
print('=' * 55)
print('GOOGLE BERT — RESULTS (5 Epochs)')
print('=' * 55)
print(f'Accuracy  : {bert_acc:.4f}  ({bert_acc*100:.2f}%)')
print(f'F1 Score  : {bert_f1:.4f}')
print(f'Eval Loss : {bert_loss:.4f}')


## Step 19 — PubMedBERT: Tokenize

In [ ]:
print('Loading PubMedBERT tokenizer ...')
pubmed_tokenizer = AutoTokenizer.from_pretrained(
    'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext'
)

def tokenize_pubmed(batch):
    return pubmed_tokenizer(
        batch['text'], truncation=True,
        padding='max_length', max_length=256
    )

pubmed_train_ds = Dataset.from_pandas(bert_train_df)
pubmed_test_ds  = Dataset.from_pandas(bert_test_df)
pubmed_train_ds = pubmed_train_ds.map(tokenize_pubmed, batched=True)
pubmed_test_ds  = pubmed_test_ds.map(tokenize_pubmed,  batched=True)
pubmed_train_ds = pubmed_train_ds.rename_column('label', 'labels')
pubmed_test_ds  = pubmed_test_ds.rename_column('label',  'labels')
cols = ['input_ids', 'attention_mask', 'token_type_ids', 'labels']
pubmed_train_ds.set_format(type='torch', columns=cols)
pubmed_test_ds.set_format( type='torch', columns=cols)
print(f'PubMedBERT tokenization done. Train: {len(pubmed_train_ds)} | Test: {len(pubmed_test_ds)}')


## Step 20 — Train PubMedBERT (5 Epochs)
> ⏱️ Same estimated time as Google BERT above.

In [ ]:
print('Loading PubMedBERT model ...')
pubmed_model = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext',
    num_labels=num_labels
)

pubmed_trainer = Trainer(
    model=pubmed_model,
    args=make_training_args('./pubmed_results'),
    train_dataset=pubmed_train_ds,
    eval_dataset=pubmed_test_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # increased from 2
)

print('Training PubMedBERT — 5 epochs ...')
pubmed_trainer.train()
print('PubMedBERT training complete!')


## Step 21 — Evaluate PubMedBERT

In [ ]:
pubmed_results = pubmed_trainer.evaluate()
pubmed_acc  = pubmed_results.get('eval_accuracy', 0)
pubmed_f1   = pubmed_results.get('eval_f1', 0)
pubmed_loss = pubmed_results.get('eval_loss', 0)
print('=' * 55)
print('PUBMEDBERT — RESULTS (5 Epochs)')
print('=' * 55)
print(f'Accuracy  : {pubmed_acc:.4f}  ({pubmed_acc*100:.2f}%)')
print(f'F1 Score  : {pubmed_f1:.4f}')
print(f'Eval Loss : {pubmed_loss:.4f}')


## Step 22 — Final Comparison: All 4 Models

In [ ]:
comparison = pd.DataFrame({
    'Model'         : ['Random Forest', 'Linear SVM', 'Google BERT', 'PubMedBERT'],
    'Accuracy'      : [rf_acc,  svm_acc,  bert_acc,  pubmed_acc],
    'F1 (weighted)' : [rf_f1,   svm_f1,   bert_f1,   pubmed_f1],
    'Eval Loss'     : [None,    None,      bert_loss, pubmed_loss]
})
print('\n' + '=' * 60)
print('FINAL MODEL COMPARISON (5-Epoch Run)')
print('=' * 60)
styled = (
    comparison.style
    .format({
        'Accuracy'      : '{:.4f}',
        'F1 (weighted)' : '{:.4f}',
        'Eval Loss'     : lambda x: f'{x:.4f}' if x is not None else 'N/A'
    })
    .highlight_max(subset=['Accuracy', 'F1 (weighted)'], color='lightgreen')
)
display(styled)


## Step 23 — Visualization: All 4 Models

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
models = comparison['Model']

for ax, metric, title in [
    (axes[0], 'Accuracy',      'Accuracy — All 4 Models'),
    (axes[1], 'F1 (weighted)', 'Weighted F1 — All 4 Models')
]:
    ax.bar(models, comparison[metric], color=colors)
    ax.set_title(title, fontsize=13)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=15)
    for i, v in enumerate(comparison[metric]):
        ax.text(i, v+0.02, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

bert_names  = ['Google BERT', 'PubMedBERT']
bert_losses = [bert_loss, pubmed_loss]
axes[2].bar(bert_names, bert_losses, color=['#C44E52', '#8172B2'])
axes[2].set_title('Eval Loss — BERT Models (lower is better)', fontsize=13)
axes[2].set_ylabel('Loss')
axes[2].tick_params(axis='x', rotation=15)
for i, v in enumerate(bert_losses):
    axes[2].text(i, v+0.01, f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


## Step 24 — Confusion Matrices: All 4 Models

In [ ]:
bert_preds_out   = bert_trainer.predict(bert_test_ds)
pubmed_preds_out = pubmed_trainer.predict(pubmed_test_ds)

bert_preds_idx    = np.argmax(bert_preds_out.predictions,   axis=-1)
pubmed_preds_idx  = np.argmax(pubmed_preds_out.predictions, axis=-1)

true_bert         = encoder.inverse_transform(bert_test_ds['labels'].numpy())
bert_pred_names   = encoder.inverse_transform(bert_preds_idx)
pubmed_pred_names = encoder.inverse_transform(pubmed_preds_idx)
label_names       = list(encoder.classes_)

fig, axes = plt.subplots(2, 2, figsize=(28, 22))
configs = [
    (y_test,    rf_pred,           'Random Forest', 'Blues',   axes[0][0]),
    (y_test,    svm_pred,          'Linear SVM',    'Greens',  axes[0][1]),
    (true_bert, bert_pred_names,   'Google BERT',   'Oranges', axes[1][0]),
    (true_bert, pubmed_pred_names, 'PubMedBERT',    'Purples', axes[1][1]),
]
for true, pred, title, cmap, ax in configs:
    cm = confusion_matrix(true, pred, labels=label_names)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=label_names, yticklabels=label_names,
                ax=ax, linewidths=0.3, annot_kws={'size': 7})
    ax.set_title(f'{title} — Confusion Matrix', fontsize=13)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)
plt.tight_layout()
plt.show()


## Step 25 — Sample Prediction Demo

In [ ]:
random.seed(99)
idx            = random.randint(0, len(X_test)-1)
sample_text    = X_test.iloc[idx]
true_label     = y_test.iloc[idx]
svm_prediction = svm_model.predict(vectorizer.transform([sample_text]))[0]
print('─' * 55)
print('SAMPLE PREDICTION DEMO (Linear SVM)')
print('─' * 55)
print(f'Report snippet   : {sample_text[:300]}...')
print(f'\nTrue cancer type : {true_label}')
print(f'SVM prediction   : {svm_prediction}')
print(f'Correct?         : {true_label == svm_prediction}')


## Step 26 — Conclusion

In [ ]:
print(f'''
=================================================================
PROJECT CONCLUSION — OPTIMIZED (5-Epoch Run)
NLP for Cancer Type Classification — TCGA Pathology Reports
=================================================================
Dataset   : {len(data)} reports | {num_labels} cancer types

── WHAT CHANGED vs 3-EPOCH VERSION ─────────────────────────────
TF-IDF     : 10,000 → 15,000 features + min_df/max_df filters
Random Forest : 200 → 300 trees + min_samples_leaf=2
SVM        : C=1.0 → C=0.5 + max_iter=5000
BERT epochs: 3 → 5
Learning rate : 2e-5 → 1e-5
Warmup     : 10% → 15%
Weight decay  : 0.01 → 0.02
Early stop patience: 2 → 3

── RESULTS ─────────────────────────────────────────────────────
Model            Accuracy      F1 (weighted)
─────────────────────────────────────────────
Random Forest    {rf_acc:.4f}        {rf_f1:.4f}
Linear SVM       {svm_acc:.4f}        {svm_f1:.4f}
Google BERT      {bert_acc:.4f}        {bert_f1:.4f}
PubMedBERT       {pubmed_acc:.4f}        {pubmed_f1:.4f}
=================================================================
''')
